# schedule data update pipeline v1

目标：验证 `src/schedule` 不只负责模型/dashboard 刷新，也能编排数据定时更新，把 raw/provider frame 对齐 qlib calendar 后发布为 canonical snapshot，并更新 data registry。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.schedule import DataUpdatePipelineConfig, run_data_update_once

EXPERIMENT_DIR = ROOT / "research" / "schedule_data_update_pipeline_v1"
DATA_DIR = EXPERIMENT_DIR / "data"
LOCK_DIR = EXPERIMENT_DIR / "locks"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

raw_frame = pd.DataFrame(
    [
        {"date": "2026-01-02", "instrument": "SOXL", "close": 10.0, "volume": 100},
        {"date": "2026-01-03", "instrument": "SOXL", "close": 11.0, "volume": 110},
        {"date": "2026-01-05", "instrument": "SOXL", "close": 12.0, "volume": 120},
    ]
)
calendar = pd.DatetimeIndex(pd.to_datetime(["2026-01-02", "2026-01-05"]), name="date")
raw_frame.to_csv(EXPERIMENT_DIR / "raw_provider_frame.csv", index=False)
raw_frame

,date,instrument,close,volume
0,2026-01-02,SOXL,10.0,100
1,2026-01-03,SOXL,11.0,110
2,2026-01-05,SOXL,12.0,120


In [2]:
events: list[str] = []
config = DataUpdatePipelineConfig(
    name="research_daily_soxl_data_update",
    dataset_key="daily.soxl_research_unit",
    dataset_name="research_unit_soxl_daily",
    universe=("SOXL",),
    output_dir=DATA_DIR / "snapshots" / "research_unit_soxl_daily" / "2026-01-05",
    registry_path=DATA_DIR / "data_registry.json",
    calendar_path="data/qlib/soxl/calendars/day.txt",
    source_paths=("research/schedule_data_update_pipeline_v1/raw_provider_frame.csv",),
    start="2026-01-01",
    end="2026-01-05",
    as_of="2026-01-05",
    published_at="2026-01-05T21:00:00Z",
    notify_details=True,
    retries=1,
)

result = run_data_update_once(
    config,
    lock_dir=LOCK_DIR,
    notify=events.append,
    frame_provider=lambda _: raw_frame,
    calendar_provider=lambda _: calendar,
)

assert result.status == "success"
assert result.payload["schema_version"] == "schedule.data_update_result.v1"
assert result.payload["row_count"] == 2
assert result.payload["dropped_non_calendar_rows"] == 1
assert result.payload["attempts"] == 1
result

JobResult(name='research_daily_soxl_data_update', status='success', payload={'schema_version': 'schedule.data_update_result.v1', 'dataset_key': 'daily.soxl_research_unit', 'dataset_name': 'research_unit_soxl_daily', 'status': 'published', 'manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_data_update_pipeline_v1/data/snapshots/research_unit_soxl_daily/2026-01-05/data_update_manifest.json', 'manifest_hash': 'a758a69b4d4c93238ae5e30cbd875168cf166e475425cb36b9a65ee2d7ae2974', 'dataset_schema_hash': '30f142ac8a3f55973b96dfc894fa1955532dd4cbb94279ac821c808608cf2ca3', 'registry_path': '/Users/rziiiii/dev/workspace/research/schedule_data_update_pipeline_v1/data/data_registry.json', 'start': '2026-01-02', 'end': '2026-01-05', 'as_of': '2026-01-05', 'row_count': 2, 'column_count': 4, 'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb', 'dropped_non_calendar_rows': 1, 'published_at': '2026-01-05T21:00:00Z', 'attempts': 1}, detail='')

In [3]:
manifest_path = Path(result.payload["manifest_path"])
registry_path = Path(result.payload["registry_path"])
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
registry = json.loads(registry_path.read_text(encoding="utf-8"))
canonical_frame = pd.read_csv(config.output_dir / "canonical_frame.csv")

assert manifest["schema_version"] == "data.update_manifest.v1"
assert manifest["manifest_hash"] == result.payload["manifest_hash"]
assert registry["datasets"][config.dataset_key]["current"]["manifest_hash"] == manifest["manifest_hash"]
assert canonical_frame["date"].tolist() == ["2026-01-02", "2026-01-05"]
assert events and "research_daily_soxl_data_update:success" in events[0]

manifest, registry["datasets"][config.dataset_key]["current"], events

({'canonical_export_schema_version': 'data.canonical_snapshot.export.v1',
  'dataset': {'calendar_path': 'data/qlib/soxl/calendars/day.txt',
   'data_paths': ['research/schedule_data_update_pipeline_v1/raw_provider_frame.csv'],
   'end': '2026-01-05',
   'metadata': {'as_of': '2026-01-05',
    'calendar_row_count': 2,
    'canonical_frame_schema_version': 'data.canonical_frame.v1',
    'column_count': 4,
    'date_column': 'date',
    'dropped_non_calendar_rows': 1,
    'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
    'row_count': 2,
    'source_paths': ['research/schedule_data_update_pipeline_v1/raw_provider_frame.csv']},
   'name': 'research_unit_soxl_daily',
   'schema_version': 'dataset.v1',
   'start': '2026-01-02',
   'universe': ['SOXL']},
  'dataset_key': 'daily.soxl_research_unit',
  'dataset_schema_hash': '30f142ac8a3f55973b96dfc894fa1955532dd4cbb94279ac821c808608cf2ca3',
  'manifest_hash': 'a758a69b4d4c93238ae5e30cbd875168cf166e475

In [4]:
summary = {
    "schema_version": "research.schedule_data_update_pipeline_v1",
    "status": result.status,
    "job_name": result.name,
    "payload": result.payload,
    "notification_events": events,
    "manifest_path": str(manifest_path.relative_to(ROOT)),
    "registry_path": str(registry_path.relative_to(ROOT)),
    "canonical_frame_rows": int(len(canonical_frame)),
    "canonical_dates": canonical_frame["date"].astype(str).tolist(),
}
(EXPERIMENT_DIR / "notification_events.json").write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8")
(EXPERIMENT_DIR / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary

{'schema_version': 'research.schedule_data_update_pipeline_v1',
 'status': 'success',
 'job_name': 'research_daily_soxl_data_update',
 'payload': {'schema_version': 'schedule.data_update_result.v1',
  'dataset_key': 'daily.soxl_research_unit',
  'dataset_name': 'research_unit_soxl_daily',
  'status': 'published',
  'manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_data_update_pipeline_v1/data/snapshots/research_unit_soxl_daily/2026-01-05/data_update_manifest.json',
  'manifest_hash': 'a758a69b4d4c93238ae5e30cbd875168cf166e475425cb36b9a65ee2d7ae2974',
  'dataset_schema_hash': '30f142ac8a3f55973b96dfc894fa1955532dd4cbb94279ac821c808608cf2ca3',
  'registry_path': '/Users/rziiiii/dev/workspace/research/schedule_data_update_pipeline_v1/data/data_registry.json',
  'start': '2026-01-02',
  'end': '2026-01-05',
  'as_of': '2026-01-05',
  'row_count': 2,
  'column_count': 4,
  'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
  'dropped_non_

## 中文结论

- 做了什么：用 `src.schedule.run_data_update_once` 编排一次 SOXL canonical data update，从 provider frame 对齐 calendar 后写出 canonical snapshot、`data_update_manifest.json` 和 `data_registry.json`。
- 核心指标：输入 3 行，calendar 对齐后保留 2 行，丢弃 1 行非交易日；manifest hash、dataset schema hash、frame schema hash 均写入 result payload 和 registry。
- regime 差异：本 notebook 聚焦数据更新编排，不做 regime 分析；后续模型/regime 仍由 `src/analysis` 读取下游 manifest/export。
- 是否晋升：否。本轮只是把 schedule 的数据更新能力补成稳定框架契约，不代表任何交易模型晋升。
- 风险：当前 provider 是 notebook 内的最小样例；生产接入真实数据源时仍需校验源许可、交易日历、复权口径、缺失值和 as_of 防未来函数。
- 下一步：把 production paper/sim replay 的 quote 选择进一步接到 data registry current snapshot，而不是直接构造局部 CSV。